# segment-line-intersect-2d — faded example 1: Extract both parameters from the solved system

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `segment-line-intersect-2d`. The last cell reports your progress on the `Geometry: Segment-line intersect 2-D` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Segment-line intersect 2-D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`segment-line-intersect-2d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "segment-line-intersect-2d"
DD_SUBTOPIC = "Geometry: Segment-line intersect 2-D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

After solving the 2×2 parametric system `[d | -e] @ [t, s]^T = L0 - S0`, the result vector contains both parameters: index 0 is the segment parameter t (must be in [0,1] for a hit), and index 1 is the line parameter s (unconstrained). You can compute the actual intersection point using either parameterization: `S0 + t*d` or `L0 + s*e`; both should give the same point.

## Faded exercise 1

Given a segment from S0=(0,0) to S1=(4,4) and an infinite line through L0=(0,3) and L1=(3,0) (the line x+y=3), find the intersection.

1. Build A and b for the system.
2. Solve with `t.linalg.solve`.
3. Extract t_seg and s_line from the solution.
4. Return `(t_seg, s_line, hit)` where hit is `0 <= t_seg <= 1`.

The blank step is extracting t_seg and s_line as Python floats from the `ts` solve result.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(0)

def seg_line_params(S0, S1, L0, L1):
    d = S1 - S0
    e = L1 - L0
    A = t.stack([d, -e], dim=1)
    b = L0 - S0
    ts = t.linalg.solve(A, b)
    t_seg, s_line = ts[0].item(), ts[1].item()
    hit = (0.0 <= t_seg <= 1.0)
    return t_seg, s_line, hit

S0 = t.tensor([0.0, 0.0])
S1 = t.tensor([4.0, 4.0])
L0 = t.tensor([0.0, 3.0])
L1 = t.tensor([3.0, 0.0])

t_seg, s_line, hit = seg_line_params(S0, S1, L0, L1)
print(f't_seg={t_seg:.4f}  s_line={s_line:.4f}  hit={hit}')
pt_from_seg  = (S0 + t_seg * (S1 - S0)).tolist()
pt_from_line = (L0 + s_line * (L1 - L0)).tolist()
print(f'Point via segment: {pt_from_seg}')
print(f'Point via line   : {pt_from_line}')


def _test():
    import torch as t

    S0 = t.tensor([0.0, 0.0])
    S1 = t.tensor([4.0, 4.0])
    L0 = t.tensor([0.0, 3.0])
    L1 = t.tensor([3.0, 0.0])

    d = S1 - S0
    e = L1 - L0
    A = t.stack([d, -e], dim=1)
    b = L0 - S0
    ts_ref = t.linalg.solve(A, b)
    expected_t = ts_ref[0].item()
    expected_s = ts_ref[1].item()

    t_seg, s_line, hit = seg_line_params(S0, S1, L0, L1)
    assert abs(t_seg - expected_t) < 1e-5, f't_seg={t_seg} expected={expected_t}'
    assert abs(s_line - expected_s) < 1e-5, f's_line={s_line} expected={expected_s}'
    assert hit == (0.0 <= expected_t <= 1.0)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

def seg_line_params(S0, S1, L0, L1):
    d = S1 - S0
    e = L1 - L0
    A = t.stack([d, -e], dim=1)
    b = L0 - S0
    ts = t.linalg.solve(A, b)
    t_seg, s_line = ts[0].item(), ts[1].item()
    hit = (0.0 <= t_seg <= 1.0)
    return t_seg, s_line, hit

S0 = t.tensor([0.0, 0.0])
S1 = t.tensor([4.0, 4.0])
L0 = t.tensor([0.0, 3.0])
L1 = t.tensor([3.0, 0.0])

t_seg, s_line, hit = seg_line_params(S0, S1, L0, L1)
print(f't_seg={t_seg:.4f}  s_line={s_line:.4f}  hit={hit}')
pt_from_seg  = (S0 + t_seg * (S1 - S0)).tolist()
pt_from_line = (L0 + s_line * (L1 - L0)).tolist()
print(f'Point via segment: {pt_from_seg}')
print(f'Point via line   : {pt_from_line}')
```
</details>